# Topic 2: Arithmetic Intensity & The Memory Wall

Welcome! We saw in Topic 1 that the GPU can compute millions of numbers in parallel. But there is a huge catch: **the data must be on the GPU to compute on it**.

In this notebook, we will investigate the hardware boundary between the CPU (Host) and GPU (Device) and understand the concept of the **Memory Wall**.

### What We Will Learn
*   **The PCIe Bottleneck:** Discover the cost of moving tensors between CPU RAM and GPU Memory.
*   **Arithmetic Intensity:** Understand the ratio of math operations to memory operations (FLOPs/byte).
*   **Host-Device Transfers:** Identify when data movement destroys your GPU performance gains.

### The Backdrop
The GPU sits on a separate physical card. It communicates with the host CPU via the **PCIe bus**. While a modern GPU's internal memory bandwidth (HBM) is incredibly wide (up to 3 Terabytes per second), the PCIe lane is a narrow straw (only 32-64 Gigabytes per second).

If you transfer data to the GPU, run a tiny operation, and transfer it right back, you are bottlenecked by the slow PCIe transfer rate. Let's measure this overhead.

### Visualizing the Memory Hierarchy

Here is a whiteboard diagram showing the bandwidth speeds at each level. Note the massive bottleneck at the PCIe junction:

![GPU Memory Hierarchy](images/gpu-memory-hierarchy.svg)

### Step 1: Let's initialize our environment

We'll import PyTorch and check if we are targeting the GPU.

In [ ]:
import time  # For wall-clock timing
import torch  # For tensor computations
device = "cuda" if torch.cuda.is_available() else "cpu"

Now let's allocate a moderately large tensor on the CPU (Host System RAM).

### Step 2: Measuring host-to-device transfer latency

We will copy a CPU tensor containing 10,000,000 floats to the GPU. We measure this transfer using our synchronization ritual.

In [ ]:
# Measure the time it takes to move data over PCIe to the GPU
x_cpu = torch.randn(10000000, device="cpu")
if device == "cuda":
    torch.cuda.synchronize()  # Clear GPU stream
start_time = time.perf_counter()  # Start CPU timer
x_gpu = x_cpu.to(device)  # Transfer tensor to GPU memory
if device == "cuda":
    torch.cuda.synchronize()  # Wait for transfer completion
print(f"Transfer time: {time.perf_counter() - start_time:.4f}s")

Observe the transfer time. It takes a significant fraction of a millisecond. Now let's compare that to the time it takes the GPU to do some math on the tensor.

### Step 3: Measuring GPU math execution speed

Now that the tensor is already on the GPU, let's time a simple element-wise addition operation.

In [ ]:
# Measure computation time on the GPU (data already on device)
if device == "cuda":
    torch.cuda.synchronize()  # Clear GPU stream
start_time = time.perf_counter()  # Start CPU timer
y_gpu = x_gpu + 1.0  # GPU math operation
if device == "cuda":
    torch.cuda.synchronize()  # Block until math completes
print(f"GPU Math time: {time.perf_counter() - start_time:.6f}s")

Notice the difference! The actual arithmetic computation took barely any time, while the transfer took significantly longer. The data movement was the bottleneck, not the math.

## First-Principles Checkpoint: The PCIe Wall

Let's look at the numbers. Transferring the data over the PCIe bus took much longer than the math itself. This ratio is governed by **Arithmetic Intensity**:

$$\text{Arithmetic Intensity} = \frac{\text{Floating Point Operations (FLOPs)}}{\text{Bytes Transferred}}$$ 

If your arithmetic intensity is low (like adding a constant to a vector), moving the vector to the GPU is a waste of time. The host-device transfer cost wipes out the speed gains.

### Rule of Thumb
To benefit from GPU acceleration, you must **minimize host-device data transfer** and **maximize computation per byte transferred**. In other words, "Keep the data on the device, move it once, and compute many times."

### Role Lens: Why it matters in practice
*   **DevOps / MLOps:** High network or PCIe transfer activity paired with low GPU compute utilization is the classic symptom of a bad dataloader. Make sure dataloading is pinned to page-locked (pinned) memory or batches are pre-fetched.
*   **Data Science:** When designing custom neural network layers, try to avoid operations that require fetching intermediate results back to the CPU (e.g., calling `.item()`, `.tolist()`, or printing tensors inside a training loop).
*   **Data Engineering:** If you build an ETL pipeline using cuDF/RAPIDS, keep the entire data transformation pipeline on the GPU and only write the final summary back to the host disk.

In our final topic for this section, we will study the ultimate GPU acceleration candidate—Matrix Multiplication—and find the physical point where the GPU starts to win.